# Análisis Exploratorio de Datos (EDA) con Claude

Template de EDA que puedes usar como prompt base para pedirle a Claude que analice cualquier dataset.

**Cómo usar este notebook:**
1. Sube tu CSV/Excel a Claude junto con este notebook como referencia
2. Pide: "Sigue la estructura de este notebook de EDA para analizar mi dataset"
3. Claude ejecutará cada sección adaptada a tus datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Carga y Vista General

In [ ]:
# Reemplazar con tu archivo
df = pd.read_csv('tu_dataset.csv')  # o pd.read_excel('archivo.xlsx')

print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nTipos de datos:')
print(df.dtypes)
print(f'\nPrimeras filas:')
df.head()

## 2. Calidad de Datos

In [ ]:
def quality_report(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Genera reporte de calidad de datos."""
    report = pd.DataFrame({
        'tipo': dataframe.dtypes,
        'nulos': dataframe.isnull().sum(),
        'pct_nulos': (dataframe.isnull().sum() / len(dataframe) * 100).round(2),
        'unicos': dataframe.nunique(),
        'duplicados': dataframe.duplicated().sum()
    })
    return report

quality_report(df)

## 3. Estadísticas Descriptivas

In [ ]:
print('=== Variables Numéricas ===')
print(df.describe().round(2))

print('\n=== Variables Categóricas ===')
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f'\n{col}: {df[col].nunique()} categorías')
    print(df[col].value_counts().head())

## 4. Distribuciones

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
n = len(num_cols)
rows = (n + 2) // 3

fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='#5B8DEF', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
    axes[i].axvline(df[col].mean(), color='#E8593C', linestyle='--', label='Media')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Correlaciones

In [ ]:
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Matriz de Correlación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlaciones fuertes
strong = corr_matrix.unstack().sort_values(ascending=False)
strong = strong[(strong < 1) & (strong.abs() > 0.5)]
if len(strong) > 0:
    print('Correlaciones fuertes (|r| > 0.5):')
    print(strong.drop_duplicates())

## 6. Outliers

In [ ]:
fig, axes = plt.subplots(1, min(len(num_cols), 5), figsize=(4 * min(len(num_cols), 5), 5))
if len(num_cols) == 1:
    axes = [axes]

for i, col in enumerate(num_cols[:5]):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#5B8DEF', alpha=0.6))
    axes[i].set_title(col)

plt.suptitle('Detección de Outliers (Boxplot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()